```yaml
title: Reading protest history before you bid
description: Pull GAO protest decisions for a target agency, sort by outcome, and pull the digests for the sustained ones to learn what went wrong.
tags: [protests, filtering, joining, proposals]
endpoints: [list_protests, get_protest]
```

# Reading protest history before you bid

Every proposal team learns the same lesson the same way: by losing a competition on a ground that GAO has already ruled on three times. Tango's `list_protests` puts the public protest record one filter away — sort by `outcome=Sustained` and you get the failure modes a contracting officer has already been corrected on, before you write the first page of your response.

## Setup

In [1]:
from collections import Counter
from tango import TangoClient

client = TangoClient()

## Pick a target agency and time window

The `agency` filter is fuzzy text against the protested-agency string, so use a phrase a GAO clerk would have typed. We'll look at the **Navy** since the start of CY25 — long enough to see patterns, short enough that the precedent is still relevant.

In [2]:
AGENCY = "Navy"
AFTER = "2025-01-01"

PROTEST_SHAPE = (
    "case_id,case_number,title,source_system,outcome,case_type,"
    "agency,protester,solicitation_number,filed_date,decision_date,decision_url"
)

protests, page = [], 1
while page <= 5:  # cap pages for CI; raise for real research
    r = client.list_protests(
        agency=AGENCY,
        filed_date_after=AFTER,
        shape=PROTEST_SHAPE,
        page=page,
        limit=100,
    )
    protests.extend(r.results)
    if not r.next:
        break
    page += 1

print(f"Pulled {len(protests)} protests against '{AGENCY}' filed since {AFTER}.")

Pulled 142 protests against 'Navy' filed since 2025-01-01.


## What's the outcome shape?

Most protests are dismissed or withdrawn — the agency takes corrective action and the protester walks away. Sustained protests are rare and signal a real procedural defect. The ratio is itself useful intel: an agency that draws a lot of dismissals may just be poking the GAO process more than usual.

In [3]:
outcomes = Counter((p.get("outcome") or "(none)") for p in protests)
total = sum(outcomes.values()) or 1
print("Outcome breakdown:")
for o, n in outcomes.most_common():
    print(f"  {n:>4}  ({n/total:>4.0%})  {o}")

Outcome breakdown:
    54  ( 38%)  Dismissed
    39  ( 27%)  Denied
    27  ( 19%)  Withdrawn
    20  ( 14%)  (none)
     2  (  1%)  Sustained


## The landmines: sustained protests

Sustained = GAO told the agency it screwed up. Each one is a small case study in what *not* to do (if you're the agency) or what's open to challenge (if you're a competitor). Pull the digests with `get_protest` — they're the GAO-authored one-paragraph summary of the holding.

In [4]:
sustained = [p for p in protests if (p.get("outcome") or "").lower() == "sustained"]
sustained.sort(key=lambda p: p.get("decision_date") or "", reverse=True)

print(f"{len(sustained)} sustained protest(s) against '{AGENCY}' since {AFTER}.\n")

# Detail-pull only what we'll print, to keep API budget small.
DETAIL_LIMIT = 5
DIGEST_SHAPE = PROTEST_SHAPE + ",digest"

for p in sustained[:DETAIL_LIMIT]:
    full = client.get_protest(p["case_id"], shape=DIGEST_SHAPE)
    # decision_date comes back as a datetime; str()[:10] gives the YYYY-MM-DD prefix
    # regardless of whether it's a date, datetime, or already-stringified value.
    decision = str(full.get("decision_date") or "")[:10]
    title = (full.get("title") or "").strip()
    protester = full.get("protester") or "?"
    sol = full.get("solicitation_number") or ""
    digest = (full.get("digest") or "").replace("\n", " ").strip()
    print(f"[{full.get('case_number')}]  {decision}  {title}")
    print(f"  protester: {protester}" + (f"  |  sol: {sol}" if sol else ""))
    if digest:
        print(f"  digest: {digest[:400]}{'…' if len(digest) > 400 else ''}")
    print()

2 sustained protest(s) against 'Navy' since 2025-01-01.



[b-423281]  2026-04-24  Owl International, Inc- dba Global 1st Flagship Company (N00024-23-R-4304)
  protester: Owl International, Inc- dba Global 1st Flagship Company  |  sol: N00024-23-R-4304
  digest: 1. Protest that agency improperly removed provision from solicitation providing for the evaluation of professional compensation is denied where the record showed a reasonable basis for the agency's determination that performance of the requirement would not require meaningful numbers of professional employees.  2. Protest that agency unreasonably restricted offerors to revising only their cost/pri…



[b-423785]  2025-12-18  Solvere Technical Group, LLC (N0017424R3002)
  protester: Solvere Technical Group, LLC  |  sol: N0017424R3002
  digest: Protest challenging evaluation of protester's proposal under personnel approach element of the technical factor and under the cost factor is sustained where record shows the evaluation was inconsistent with the solicitation.



## Who keeps protesting here

Repeat protesters tell you who is willing to fight at this agency. If you're competing against one of them, your proposal needs to be airtight on the grounds they've previously raised.

In [5]:
repeat = Counter(p.get("protester") for p in protests if p.get("protester"))
print("Most frequent protesters in this window:")
for name, n in repeat.most_common(10):
    print(f"  {n:>3}  {name}")

Most frequent protesters in this window:
    4  JDSAT Inc.
    3  Diversified Maintenance Systems, Inc.
    2  Secise LLC
    2  Schuyler Line Navigation Company, LLC
    2  SeaCorp LLC
    2  ASG Solutions Corporation- dba American Systems Group
    2  SPARC JV, LLC
    2  ARiA
    2  Island Creek Associates, LLC
    2  C4CJV, LLC
